# 04 Compare Results

This notebook aggregates baseline and finetuned reports and writes a final summary to `outputs/notebooks/comparison/summary.json`.

In [ ]:
from pathlib import Path
import json
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/abstention-data")

if not (ROOT / "data" / "dataset1.jsonl").exists() or not (ROOT / "src").exists():
    raise FileNotFoundError(
        f"Invalid ROOT: {ROOT}. Make sure repo is in Drive at /content/drive/MyDrive/abstention-data"
    )

BASELINE_DIR = ROOT / "outputs" / "notebooks" / "baseline"
LORA1_DIR = ROOT / "outputs" / "notebooks" / "lora_dataset1" / "eval"
LORA2_DIR = ROOT / "outputs" / "notebooks" / "lora_dataset2" / "eval"
OUT_DIR = ROOT / "outputs" / "notebooks" / "comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)

In [ ]:
def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

baseline_summary = load_json(BASELINE_DIR / "summary.json")
lora1_report = load_json(LORA1_DIR / "dataset4.json")
lora2_report = load_json(LORA2_DIR / "dataset4.json")

if baseline_summary is None:
    raise FileNotFoundError(f"Missing baseline summary: {BASELINE_DIR / 'summary.json'}")
if lora1_report is None:
    raise FileNotFoundError(f"Missing lora_dataset1 eval report: {LORA1_DIR / 'dataset4.json'}")
if lora2_report is None:
    raise FileNotFoundError(f"Missing lora_dataset2 eval report: {LORA2_DIR / 'dataset4.json'}")

comparison = {
    "baseline_on_4": baseline_summary["datasets"].get("dataset4", {}),
    "train_on_1_eval_on_4": lora1_report.get("metrics", {}),
    "train_on_2_eval_on_4": lora2_report.get("metrics", {}),
}

In [ ]:
summary_path = OUT_DIR / "summary.json"
summary_path.write_text(json.dumps(comparison, indent=2), encoding="utf-8")
print("Saved:", summary_path)
print(json.dumps(comparison, indent=2))

In [ ]:
# Compact table view
rows = [
    ("baseline_on_4", comparison["baseline_on_4"]),
    ("train_on_1_eval_on_4", comparison["train_on_1_eval_on_4"]),
    ("train_on_2_eval_on_4", comparison["train_on_2_eval_on_4"]),
]

metrics_to_show = [
    "overall_exact_match",
    "answerable_exact_match",
    "abstain_precision",
    "abstain_recall",
    "abstain_f1",
    "pred_abstain_rate",
    "gold_abstain_rate",
]

for name, metric_dict in rows:
    print(f"\n{name}")
    for key in metrics_to_show:
        print(f"  {key}: {metric_dict.get(key)}")